# Metrics for predicting numbers (regression)

_Metrics & Evaluation — measuring models, data & statistics_

**Your model guesses a number. These scores say how far off it usually is.**

Every regression score starts from one number: the residual, the gap between truth and guess, $e_i = y_i - \hat{y}_i$. Here $y_i$ is the true value for example $i$, $\hat{y}_i$ (read "y-hat") is the model's prediction, and a residual of $+3$ means the truth was $3$ above the guess.

       A whole test set gives you a pile of residuals. A metric is just a recipe for boiling that pile down to one summary number. The recipes differ in how they treat a big error versus a small one, and whether they care about absolute size ($\$5$) or relative size ($5\%$).

---

This notebook is a practice scaffold for the **Metrics for predicting numbers (regression)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Train a model and get its residualsEvery regression metric starts from the **residuals** — the gaps between the true value and the prediction. So first we split the diabetes data, standardise the features, and fit a Ridge model, then predict on the held-out test set.All the scores below are just different recipes for summarising the pile of test residuals into one number.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, HuberRegressor, QuantileRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, root_mean_squared_error,
    mean_absolute_percentage_error, r2_score, median_absolute_error,
    max_error, mean_pinball_loss, mean_tweedie_deviance)

X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.33, random_state=0)

model = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_tr, y_tr)
pred = model.predict(X_te)

### Step 2 — Error-size metrics in real unitsThese scores all live in the target's own units (here, disease-progression points). They differ in how they treat a big miss versus a small one.**MAE** averages the absolute errors (every point counts equally); **MSE** squares them so outliers dominate; **RMSE** square-roots MSE to get back into target units; **medAE** takes the *median* error so outliers are ignored; **maxE** reports the single worst miss.

In [ ]:
# Each lives in the target's units; they differ in how outliers are treated.
print("MAE  :", mean_absolute_error(y_te, pred))            # average |miss|, robust-ish
print("MSE  :", mean_squared_error(y_te, pred))             # squared -> outliers dominate
print("RMSE :", root_mean_squared_error(y_te, pred))        # back in target units (sklearn>=1.4)
print("medAE:", median_absolute_error(y_te, pred))          # middle error, ignores outliers
print("maxE :", max_error(y_te, pred))                      # worst single miss

### Step 3 — Relative and fraction-explained metricsSometimes you care about *relative* error (5%) rather than absolute ($5). **MAPE** divides each error by the truth — but blows up when `y` is near zero. **WAPE** divides *total* error by *total* truth (zero-safe), and **sMAPE** is a symmetric variant.The **R²** family asks what fraction of the variance the model explains; it can go negative on test data. **Adjusted R²** penalises adding features, and **mean bias** is the signed average error — which way the model leans.

In [ ]:
# --- percentage / relative family (these break near zero!) ---
print("MAPE :", mean_absolute_percentage_error(y_te, pred))  # fraction; blows up if y~0

# WAPE: total error / total truth -- zero-safe, computed by hand
total_error = np.abs(y_te - pred).sum()
total_truth = np.abs(y_te).sum()
wape = total_error / total_truth
print("WAPE :", wape)

# sMAPE: symmetric percentage error, also by hand
symmetric_denominator = (np.abs(y_te) + np.abs(pred)) / 2
smape = (np.abs(y_te - pred) / symmetric_denominator).mean()
print("sMAPE:", smape)

# --- fraction-explained family ---
r2 = r2_score(y_te, pred)
print("R2       :", r2)                                      # can be NEGATIVE on test data

n, p = X_te.shape
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("adj. R2  :", adj_r2)                                  # penalizes extra features

# --- mean bias: signed average error (which way are we off?) ---
mean_bias = np.mean(pred - y_te)
print("mean bias:", mean_bias)                               # + = over-predict

### Step 4 — Robust and asymmetric training lossesSome losses are used to *train* models that handle outliers or one-sided risk. **Huber** is squared near zero but linear in the tails, so a few huge errors don't dominate the fit. **Quantile (pinball) loss** lets you predict a quantile — here the 0.9 quantile for an upper bound.**RMSLE** is RMSE computed on `log1p`-transformed values, which measures *relative* error and is gentle on large targets.

In [ ]:
# Huber: squared near 0, linear in the tails -> an outlier-resistant fit
huber = make_pipeline(StandardScaler(), HuberRegressor()).fit(X_tr, y_tr)

# Quantile (pinball) loss: predict the 0.9 quantile for an upper bound
q90 = make_pipeline(
    StandardScaler(),
    QuantileRegressor(quantile=0.9, alpha=0.0, solver="highs")).fit(X_tr, y_tr)
pinball = mean_pinball_loss(y_te, q90.predict(X_te), alpha=0.9)
print("pinball@0.9:", pinball)

# RMSLE: RMSE of log1p-transformed targets (relative error, gentle on large values)
y_te_log = np.log1p(np.maximum(y_te, 0))
pred_log = np.log1p(np.maximum(pred, 0))
rmsle = root_mean_squared_error(y_te_log, pred_log)
print("RMSLE:", rmsle)

## Visualize the data & results

_On the real diabetes dataset, how close do a Ridge model's predictions land to the actual disease-progression values?_

### Step 1 — Fit the model and score itTo see how close predictions land, we re-fit the same standardised Ridge model on the diabetes data and predict on the test set. We then report three headline scores: **MAE** and **RMSE** in target units, and **R²** for the fraction of variance explained.Keeping the fit self-contained here means this cell runs on its own.

In [ ]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (mean_absolute_error,
    root_mean_squared_error, r2_score)

X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.33, random_state=0)

model = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(X_tr, y_tr)
pred = model.predict(X_te)

print("MAE :", round(mean_absolute_error(y_te, pred), 2))     # 44.96
print("RMSE:", round(root_mean_squared_error(y_te, pred), 2))  # 56.09
print("R2  :", round(r2_score(y_te, pred), 3))                # 0.403

### Step 2 — Prepare the predicted-vs-actual pointsA scatter of predicted against actual values is the clearest picture of fit: points on the `y = x` line are perfect predictions. To keep the plot readable we sort by the true value and sample about 55 evenly-spaced points.We also record the span of the `y = x` reference line so it can be drawn across the full range.

In [ ]:
# Sort by true value, then sample ~55 evenly-spaced points for a readable scatter.
order = np.argsort(y_te)
step = max(1, len(order) // 55)
sel = order[::step][:55]

pts = np.column_stack([y_te[sel], pred[sel]])
print("plotted (actual, predicted) points:", np.round(pts, 1).tolist())

# The y = x line (perfect prediction) spans from the min actual to the max of either axis.
line_lo = float(y_te.min())
line_hi = float(max(y_te.max(), pred.max()))
print("y=x line spans:", [line_lo, line_hi])

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Two house-price models are tested. Model A has MAE $\$18$k and RMSE $\$22$k. Model B has MAE $\$16$k and RMSE $\$41$k. Which would you trust more for a typical house, and what does the RMSE–MAE gap tell you about Model B?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare MAE first. — _MAE is the typical miss in dollars. Model B's is lower ($\$16$k vs $\$18$k), so on a normal house B is slightly better on average._
- Look at the gap between RMSE and MAE. — _RMSE is always $\ge$ MAE; a large gap means a few huge errors are inflating RMSE. A's gap is small ($\$4$k); B's is huge ($\$25$k)._
- Decide based on what you fear. — _B's big RMSE–MAE gap says it occasionally misses a house by a fortune. If a single catastrophic miss is unacceptable, prefer A despite its higher average._

**Answer:** For a typical house, Model B is marginally better (lower MAE). But B's RMSE ($\$41$k) towers over its MAE ($\$16$k), revealing a handful of catastrophic misses that A doesn't have (A's RMSE and MAE are close). If big errors are costly — and in pricing they usually are — trust Model A. Always report MAE and RMSE: the gap between them is the outlier signal.

</details>

**Problem 2.** A demand forecaster reports MAPE = 8% on most products but a single new product (true demand often $0$ or $1$ unit) shows MAPE = 4000%, dragging the company average up. What's going wrong, and which metric should they switch to?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall MAPE's formula. — _MAPE divides each error by the true value $y_i$. When $y_i$ is near zero, even a tiny absolute error becomes an enormous percentage._
- Spot the near-zero targets. — _The new product's demand sits at $0$–$1$ unit; an error of $2$ units is $200\%$ there, and a true $0$ makes MAPE divide by zero entirely._
- Pick a zero-safe relative metric. — _WAPE divides total error by total demand (never an individual $y_i$), and MASE divides by a naive baseline's error — both survive zeros and small values._

**Answer:** MAPE is breaking near zero: dividing by the new product's tiny demand turns small absolute errors into thousands of percent. The metric is misleading, not the model. Switch to WAPE (total error ÷ total demand — one stable percentage for the whole catalog) or MASE (error scaled by a naive forecast), both of which handle zeros and let you compare products on different scales.

</details>

**Problem 3.** On a held-out test set, your regressor scores $R^2 = -0.15$. A teammate says "negative R² must be a bug — it's a probability, it can't go below zero." Are they right?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall what R² actually compares. — _$R^2 = 1 - \frac{\text{your squared error}}{\text{error of always guessing the mean}}$. It is not a probability; it can be any value $\le 1$._
- Reason about the negative case. — _If your model's squared error exceeds the mean-baseline's error, the ratio is above $1$, so $R^2$ drops below $0$. On unseen data this is entirely possible._
- Interpret the warning. — _$R^2 = -0.15$ means the model predicts the test set worse than just outputting the average — a sign of overfitting, leakage, or distribution shift._

**Answer:** The teammate is wrong. $R^2$ is not a probability and legitimately goes negative on held-out data: it just means your model has more squared error than the dumb "always predict the mean" baseline. $R^2 = -0.15$ is a real, useful warning that the model is worse than guessing the average — investigate overfitting, leakage, or a shifted test distribution, not a code bug.

</details>